In [194]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_auc_score, roc_curve, precision_recall_curve, auc, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PowerTransformer, PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from skopt.space import Real, Integer, Categorical
from scipy.stats import randint, uniform
from skopt import BayesSearchCV
from sklearn.pipeline import Pipeline
from category_encoders import BinaryEncoder, TargetEncoder, WOEEncoder

import optuna

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, ElasticNet



In [195]:
train = ('/kaggle/input/recruitment-task-for-gdsc-ml/MiNDAT.csv', '/root/.cache/kagglehub/competitions/recruitment-task-for-gdsc-ml/MiNDAT.csv', 'Data/mindat.csv')
test = ('/kaggle/input/recruitment-task-for-gdsc-ml/MiNDAT_UNK.csv', '/root/.cache/kagglehub/competitions/recruitment-task-for-gdsc-ml/MiNDAT_UNK.csv', 'data/mindat_unk.csv')

In [196]:
df = pd.read_csv(train[2], index_col='LOCAL_IDENTIFIER')
df_test = pd.read_csv(test[2], index_col='LOCAL_IDENTIFIER')

In [197]:
df

,Z~x0<k,"vzo.""",+U@,A>.,hp!,>?64:,@wnsk>R,"U""r",&%)LTaWRb,r1Ng,...,v0rt3X,^%a;,b1oRb13,v1rt3X,0HU2N='U,ZrK,.6AvGp,3I\y,b2oRb13,CORRUCYSTIC_DENSITY
LOCAL_IDENTIFIER,,,,,,,,,,,,,,,,,,,,,
0,0.896836,-0.916578,0.632930,0.632930,-0.186669,0.422301,-0.908760,0.355945,-0.116258,0.388952,...,0.874539,-0.146314,-8.658635,0.672180,0.940507,0.117237,0.655947,-0.643482,-7.553254,521.443834
1,0.785652,-0.031439,0.265600,0.265600,NaN,0.636177,0.324519,0.933113,-0.975176,-0.529054,...,NaN,0.817420,5.426208,1.009423,-0.926698,0.974193,1.099751,0.026357,1.097666,373.696925
2,0.294313,-0.135393,0.950750,0.950750,-0.447654,0.953560,-0.178530,0.239216,0.390651,0.319620,...,1.888184,0.115265,4.255919,0.029037,NaN,3.446213,-1.007350,0.806074,1.858609,398.622235
3,0.497446,-0.049535,0.141023,0.141023,1.194801,0.000026,-1.032371,0.818995,NaN,0.031635,...,0.100983,0.907092,4.645116,-0.255313,1.797700,1.005194,1.278741,-0.818691,2.812549,527.945234
4,0.605701,1.453710,0.126049,0.126049,-0.635441,0.600073,0.776764,0.907868,-0.180334,0.662187,...,-0.894951,NaN,-1.337387,0.393958,-0.373289,0.427955,-0.726291,1.274686,10.512384,575.052682
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,NaN,-0.837980,0.857335,0.857335,-0.900921,0.305825,-1.759959,0.379284,-0.431987,-0.534191,...,0.481267,NaN,6.246344,1.023857,-0.428366,NaN,0.667137,-0.308231,2.791238,235.995368
11996,0.881287,1.647118,NaN,0.239317,-1.544021,0.205085,-0.387830,0.252647,0.550154,0.167350,...,1.700230,-1.311655,-8.038122,0.146051,1.340027,-0.228400,-0.535165,-0.142828,-7.382469,421.925749
11997,0.061169,-0.483820,0.879857,NaN,-0.161385,0.914789,-1.050923,0.051287,0.019750,0.173550,...,1.761554,-1.122804,3.745146,-0.132992,-0.812099,-1.522920,-0.115161,0.182573,2.058194,206.700030


In [198]:
df.isnull().sum()

Z~x0<k                 1009
vzo."                   964
+U@                     971
A>.                     945
hp!                     990
>?64:                   929
@wnsk>R                 919
U"r                    1004
&%)LTaWRb               963
r1Ng                    969
|G}                     962
TSWm                    981
r2Ng                   1000
maT_r                   962
@V9                     938
T\!                     954
14W$Q                  1007
ZZw3=!t                 987
.o<m                    965
.b6nl                   907
<!!                     986
F'3Ku                   954
~7*                     963
MINDSPIKE_VERSION      1951
9Z/5)2                  950
%IiL7w                  939
!;@Jw                   997
fPqsI                   974
ZVf                     939
i]7V                    936
Jv[i                    961
;<"<i(T                 890
Kj,                     973
w-u:jN'qI               914
PZ8                     978
jNhEum              

In [199]:
df = df.dropna(subset=['CORRUCYSTIC_DENSITY']).copy()
df

,Z~x0<k,"vzo.""",+U@,A>.,hp!,>?64:,@wnsk>R,"U""r",&%)LTaWRb,r1Ng,...,v0rt3X,^%a;,b1oRb13,v1rt3X,0HU2N='U,ZrK,.6AvGp,3I\y,b2oRb13,CORRUCYSTIC_DENSITY
LOCAL_IDENTIFIER,,,,,,,,,,,,,,,,,,,,,
0,0.896836,-0.916578,0.632930,0.632930,-0.186669,0.422301,-0.908760,0.355945,-0.116258,0.388952,...,0.874539,-0.146314,-8.658635,0.672180,0.940507,0.117237,0.655947,-0.643482,-7.553254,521.443834
1,0.785652,-0.031439,0.265600,0.265600,NaN,0.636177,0.324519,0.933113,-0.975176,-0.529054,...,NaN,0.817420,5.426208,1.009423,-0.926698,0.974193,1.099751,0.026357,1.097666,373.696925
2,0.294313,-0.135393,0.950750,0.950750,-0.447654,0.953560,-0.178530,0.239216,0.390651,0.319620,...,1.888184,0.115265,4.255919,0.029037,NaN,3.446213,-1.007350,0.806074,1.858609,398.622235
3,0.497446,-0.049535,0.141023,0.141023,1.194801,0.000026,-1.032371,0.818995,NaN,0.031635,...,0.100983,0.907092,4.645116,-0.255313,1.797700,1.005194,1.278741,-0.818691,2.812549,527.945234
4,0.605701,1.453710,0.126049,0.126049,-0.635441,0.600073,0.776764,0.907868,-0.180334,0.662187,...,-0.894951,NaN,-1.337387,0.393958,-0.373289,0.427955,-0.726291,1.274686,10.512384,575.052682
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,NaN,-0.837980,0.857335,0.857335,-0.900921,0.305825,-1.759959,0.379284,-0.431987,-0.534191,...,0.481267,NaN,6.246344,1.023857,-0.428366,NaN,0.667137,-0.308231,2.791238,235.995368
11996,0.881287,1.647118,NaN,0.239317,-1.544021,0.205085,-0.387830,0.252647,0.550154,0.167350,...,1.700230,-1.311655,-8.038122,0.146051,1.340027,-0.228400,-0.535165,-0.142828,-7.382469,421.925749
11997,0.061169,-0.483820,0.879857,NaN,-0.161385,0.914789,-1.050923,0.051287,0.019750,0.173550,...,1.761554,-1.122804,3.745146,-0.132992,-0.812099,-1.522920,-0.115161,0.182573,2.058194,206.700030


In [200]:
df.drop_duplicates(inplace=True)
df_test.drop_duplicates(inplace=True)
X_test = df_test.copy()

In [201]:
X = df.drop('CORRUCYSTIC_DENSITY', axis=1)
y = df['CORRUCYSTIC_DENSITY']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

num = list(X_train.select_dtypes(exclude='object').columns)
obj = list(X_train.select_dtypes(include='object').columns)

In [202]:
for i in obj:
    print(df[i].isnull().sum())


882
883
1793


In [203]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer(method='yeo-johnson')),
    ('std_scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num),
    ('cat', cat_pipe, obj)
])

In [204]:
def objective(trial):
    model_name = trial.suggest_categorical('model', ['xgboost', 'lightgbm', 'random_forest', 'elastic_net'])
    
    if model_name == 'xgboost':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
            'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)
        }
        model = XGBRegressor(**params, random_state=42, n_jobs=-1)
    
    elif model_name == 'lightgbm':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 200),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)
        }
        model = LGBMRegressor(**params, random_state=42, n_jobs=-1, verbose=-1)
    
    elif model_name == 'random_forest':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'max_depth': trial.suggest_int('max_depth', 5, 25),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2'])
        }
        model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
        
    elif model_name == 'elastic_net':
        params = {
            'alpha': trial.suggest_float('alpha', 1e-4, 1.0, log=True),
            'l1_ratio': trial.suggest_float('l1_ratio', 0.1, 1.0)
        }
        model = ElasticNet(**params, random_state=42)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)
    
    rmse = np.sqrt(np.mean((y_val - y_pred) ** 2))
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=1000)

[I 2025-08-26 13:32:54,806] A new study created in memory with name: no-name-0b31ed22-f782-46ba-9d13-e07baf0c2776
[I 2025-08-26 13:32:55,309] Trial 0 finished with value: 187.1002467516991 and parameters: {'model': 'elastic_net', 'alpha': 0.009163906812582877, 'l1_ratio': 0.6053586195719215}. Best is trial 0 with value: 187.1002467516991.
[I 2025-08-26 13:32:58,000] Trial 1 finished with value: 183.3461984607448 and parameters: {'model': 'random_forest', 'n_estimators': 609, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 13, 'max_features': 'sqrt'}. Best is trial 1 with value: 183.3461984607448.
[I 2025-08-26 13:32:58,470] Trial 2 finished with value: 187.0815986423316 and parameters: {'model': 'elastic_net', 'alpha': 0.008970914620468616, 'l1_ratio': 0.47573264280248484}. Best is trial 1 with value: 183.3461984607448.
[I 2025-08-26 13:32:59,696] Trial 3 finished with value: 175.59433286935783 and parameters: {'model': 'lightgbm', 'n_estimators': 533, 'learning_rate': 0.

KeyboardInterrupt: 

In [205]:
best_model_name = study.best_trial.params['model']
print(f"Best model: {best_model_name}")
best_params = {k: v for k, v in study.best_trial.params.items() if k != 'model'}
if best_model_name == 'xgboost':
    best_model = XGBRegressor(**best_params, random_state=42)
elif best_model_name == 'lightgbm':
    best_model = LGBMRegressor(**best_params, random_state=42)
elif best_model_name == 'random_forest':
    best_model = RandomForestRegressor(**best_params, random_state=42)
else:
    best_model = ElasticNet(**best_params, random_state=42)
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', best_model)
])
model.fit(X_train, y_train)
model.score(X_val, y_val)

Best model: lightgbm


  File "c:\Users\dsapu\Project\conda\condasystem\envs\dmsdl\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\dsapu\Project\conda\condasystem\envs\dmsdl\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\dsapu\Project\conda\condasystem\envs\dmsdl\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\dsapu\Project\conda\condasystem\envs\dmsdl\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


[LightGBM] [Warning] feature_fraction is set=0.8032560369868035, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8032560369868035
[LightGBM] [Warning] bagging_fraction is set=0.8569100979234635, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8569100979234635
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8032560369868035, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8032560369868035
[LightGBM] [Warning] bagging_fraction is set=0.8569100979234635, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8569100979234635
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002197 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] To

0.1988233564902443

In [206]:
predictions = model.predict(X_test)
submission = pd.DataFrame({
    'LOCAL_IDENTIFIER': X_test.index,
    'CORRUCYSTIC_DENSITY': predictions
})
submission.to_csv('submission.csv', index=False)


[LightGBM] [Warning] feature_fraction is set=0.8032560369868035, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8032560369868035
[LightGBM] [Warning] bagging_fraction is set=0.8569100979234635, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8569100979234635
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1


# dataset and dataloader

In [ ]:
# import torch.nn as nn
# import torch
# import torch.nn.functional as F
# import torch.optim as optim
# from torch.utils.data import DataLoader, TensorDataset
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from tqdm import tqdm

# import pytorch_lightning as pl
# import pytorch_optimizer as optim1
# from torchmetrics.regression import MeanSquaredError, R2Score
# from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
# import torchmetrics
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device

In [ ]:
# X_train = torch.tensor(X_train, dtype=torch.float32)
# y_train = torch.tensor(y_train.values, dtype=torch.float32)

# X_val = torch.tensor(X_val, dtype=torch.float32)
# y_val = torch.tensor(y_val.values, dtype=torch.float32)

In [ ]:
# train_data = TensorDataset(X_train, y_train)
# train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
# val_data = TensorDataset(X_val, y_val)
# val_loader = DataLoader(val_data, batch_size=64, shuffle=False)

In [ ]:
# def conv_block(in_feature, out_feature, padding=1, stride=1,
#              activation="relu", pool =True, maxpool=True, kernel_size=3,
#              kernel_size_pool=2, pool_stride=2)-> list[nn.Sequential]:
#     layers = [nn.Conv2d(in_feature, out_feature, kernel_size=kernel_size, padding=padding, stride=stride)]
#     if activation == "relu":
#         layers.append(nn.ReLU())
#     elif activation == "leakyrelu":
#         layers.append(nn.LeakyReLU())
#     elif activation == "sigmoid":
#         layers.append(nn.Sigmoid())
#     elif activation == "tanh":
#         layers.append(nn.Tanh())
#     if pool:
#         if maxpool:
#             layers.append(nn.MaxPool2d(kernel_size=kernel_size_pool, stride=pool_stride))
#         else:
#             layers.append(nn.AvgPool2d(kernel_size=kernel_size_pool, stride=pool_stride))
#     else:
#         layers.append(nn.Identity())
#     return nn.Sequential(*layers)


# import torch.nn as nn

# def linear_block(in_features, out_features, batch_norm=False, activation=None, dropout=0.0):
#     """
#     Membuat blok sekuensial yang terdiri dari Linear, BatchNorm (opsional), 
#     Aktivasi (opsional), dan Dropout (opsional).
#     """
#     layers = [nn.Linear(in_features, out_features)]
#     if batch_norm:
#         layers.append(nn.BatchNorm1d(out_features))
#     if activation == 'relu':
#         layers.append(nn.ReLU())
#     elif activation == 'sigmoid':
#         layers.append(nn.Sigmoid())
#     elif activation == 'tanh':
#         layers.append(nn.Tanh())
#     elif activation == 'leakyrelu':
#         layers.append(nn.LeakyReLU(0.1)) 
#     elif activation == 'softmax':
#         layers.append(nn.Softmax(dim=1))
#     elif activation == 'elu':
#         layers.append(nn.ELU())
#     elif activation == 'selu':
#         layers.append(nn.SELU())
#     elif activation == 'lsoftmax':
#         layers.append(nn.LogSoftmax(dim=1))
        
#     if dropout > 0.0:
#         layers.append(nn.Dropout(dropout))
        
#     return nn.Sequential(*layers)

In [ ]:
# class ANN(pl.LightningModule):
#     def __init__(self, input_size, learning_rate, pos_weight_tensor, dropout=0):
#         super(ANN, self).__init__()
#         self.learning_rate = learning_rate
#         self.save_hyperparameters()
#         self.criterion = nn.MSELoss()
#         self.mae = MeanSquaredError()
#         self.r2_score = R2Score()
#         self.fc = nn.Sequential(
#             linear_block(input_size, 128, activation='leakyrelu', batch_norm=True),
#             linear_block(128, 64, activation='leakyrelu', batch_norm=True),
#             linear_block(64, 32, activation='leakyrelu', batch_norm=True),
#             linear_block(32, 1, activation=None)
#         )
#     def forward(self, x):
#         return self.fc(x)

#     def training_step(self, batch, batch_idx):
#         inputs, labels = batch
#         labels = labels.unsqueeze(1) 
#         outputs = self(inputs)
#         loss = self.criterion(outputs, labels)
#         mae = self.mae(outputs, labels)
#         r2 = self.r2_score(outputs, labels)
#         self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
#         self.log('train_mae', mae, on_step=False, on_epoch=True, prog_bar=True, logger=True)
#         self.log('train_r2', r2, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        
#         return loss

#     def validation_step(self, batch, batch_idx):
#         inputs, labels = batch
#         labels = labels.unsqueeze(1)
#         outputs = self(inputs)
#         loss = self.criterion(outputs, labels)
#         mae = self.mae(outputs, labels)
#         r2 = self.r2_score(outputs, labels)
#         self.log('val_loss', loss, on_epoch=True, prog_bar=True, logger=True)
#         self.log('val_mae', mae, on_epoch=True, prog_bar=True, logger=True)
#         self.log('val_r2', r2, on_epoch=True, prog_bar=True, logger=True)

#     def test_step(self, batch, batch_idx):
#         inputs, labels = batch
#         labels = labels.unsqueeze(1)
#         outputs = self(inputs)
#         loss = self.criterion(outputs, labels)
#         mae = self.mae(outputs, labels)
#         r2 = self.r2_score(outputs, labels)
#         self.log('test_loss', loss, on_epoch=True, prog_bar=True, logger=True)
#         self.log('test_mae', mae, on_epoch=True, prog_bar=True, logger=True)
#         self.log('test_r2', r2, on_epoch=True, prog_bar=True, logger=True)

#     def backward(self, loss, *args, **kwargs):
#         loss.backward(create_graph=True)

#     def configure_optimizers(self):
#         optimizer = optim1.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=1e-4)
#         return optimizer

#     def predict_step(self, batch, batch_idx, dataloader_idx=None):
#         if isinstance(batch, list) or isinstance(batch, tuple):
#             inputs, _ = batch
#         else:
#             inputs = batch
#         return self(inputs)

In [ ]:
# X_train.shape[1]

# Config

In [ ]:
# if torch.cuda.is_available():
#     accelerator_type = 'gpu'
#     devices_to_use = 1
# else:
#     accelerator_type = 'cpu'
#     devices_to_use = 'auto'

# checkpoint_callback = ModelCheckpoint(
#     monitor='val_r2',
#     dirpath='checkpoints/',
#     filename='GDSC-{epoch:02d}-{val_loss:.4f}',
#     save_top_k=1,
#     mode='max'
# )
# early_stopping = EarlyStopping(
#     monitor='val_r2',
#     patience=10,
#     mode='max',
# )
# lr_monitor_callback = LearningRateMonitor(logging_interval='epoch')

# trainer1 = pl.Trainer(
#     max_epochs=3000,
#     callbacks=[checkpoint_callback, early_stopping, lr_monitor_callback],
#     logger=pl.loggers.TensorBoardLogger("tb_logs", name="simple_model_experiment"),
#     accelerator=accelerator_type,
#     devices=devices_to_use,
#     log_every_n_steps=10,
#     gradient_clip_val=0.5,
#     deterministic=True
# )

# model = ANN(input_size=X_train.shape[1], learning_rate=0.0005, pos_weight_tensor=None, dropout=0)

In [ ]:
# from pytorch_lightning.tuner import Tuner
# print("--- Menjalankan Learning Rate Finder ---")
# # Buat objek Tuner dari trainer Anda
# tuner = Tuner(trainer1)

# # Jalankan lr_find
# lr_finder = tuner.lr_find(model, train_loader, val_loader)

# # Ambil learning rate yang disarankan
# suggested_lr = lr_finder.suggestion()
# print(f"Saran learning rate dari tuner: {suggested_lr}")

# # Perbarui learning rate di model Anda
# model.learning_rate = suggested_lr
# # atau jika Anda menggunakan hparams: model.hparams.learning_rate = suggested_lr

# # Tampilkan plot untuk dianalisis (opsional)
# fig = lr_finder.plot(suggest=True)
# fig.show()

In [ ]:
# trainer1.fit(model, train_loader, val_loader)

In [ ]:
# trainer1.validate(model, val_loader, ckpt_path='best')
